In [ ]:
import pandas as pd
import random
import numpy as np
from random import randint

import pickle
import time
import memory_profiler

%load_ext memory_profiler

from pathlib import Path
import distro

%load_ext watermark

The memory_profiler extension is already loaded. To reload it, use:
  %reload_ext memory_profiler
The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark


In [ ]:
%load_ext autoreload
%autoreload 2

from text_embeddings_src.legacy.metrics import knn_accuracy_whitening_scores

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import black
import jupyter_black

jupyter_black.load(line_length=79)

In [ ]:
variables_path = Path("../results/variables")
figures_path = Path("../results/figures/updated_dataset")
data_path = Path("../data")

In [ ]:
# MANUAL FIX TO PATH ISSUE FROM VSCODE
import text_embeddings_src

nb_path = Path(text_embeddings_src.__path__[0]).parents[0] / Path("scripts")
assert nb_path.exists(), "The path does not exist"

variables_path = (nb_path / variables_path).resolve(strict=True)
figures_path = (nb_path / figures_path).resolve(strict=True)
data_path = (nb_path / data_path).resolve(strict=True)

In [ ]:
%watermark -a 'Rita González-Márquez' -t -d -tz -u -v -iv -w -m -h -p transformers -p openTSNE
print(distro.name(pretty=True))

Author: Rita González-Márquez

Last updated: 2026-02-24 14:31:12CET

Python implementation: CPython
Python version       : 3.12.4
IPython version      : 8.31.0

openTSNE: 1.0.2

Compiler    : GCC 11.2.0
OS          : Linux
Release     : 5.14.0-570.17.1.el9_6.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 64
Architecture: 64bit

Hostname: rgonzalesmarquez_GPU0-llm_gber7

text_embeddings_src: 0.0.0
distro             : 1.9.0
numpy              : 1.26.4
black              : 24.10.0
jupyter_black      : 0.4.0
pandas             : 2.2.3
memory_profiler    : 0.61.0

Watermark: 2.5.0

Ubuntu 24.04 LTS


# Import

In [ ]:
%%time
iclr2024 = pd.read_parquet(
    data_path / "iclr2024.parquet.gzip",
    engine="pyarrow",
)

CPU times: user 256 ms, sys: 84.8 ms, total: 341 ms
Wall time: 363 ms


In [ ]:
iclr2024.keywords = iclr2024.keywords.transform(lambda x: list(x))
iclr2024.scores = iclr2024.scores.transform(lambda x: list(x))

In [ ]:
iclr2024

,index,year,id,title,abstract,authors,decision,scores,keywords,gender-first,gender-last,t-SNE x,t-SNE y
0,0,2017,S1VaB4cex,FractalNet: Ultra-Deep Neural Networks without...,We introduce a design strategy for neural netw...,"Gustav Larsson, Michael Maire, Gregory Shakhna...",Accept (Poster),"[5, 7, 6, 6]",[],male,male,-28.117955,-20.418127
1,1,2017,H1W1UN9gg,Deep Information Propagation,We study the behavior of untrained neural netw...,"Samuel S. Schoenholz, Justin Gilmer, Surya Gan...",Accept (Poster),"[8, 9, 8]","[theory, deep learning]",male,None,-32.466820,-10.791123
2,2,2017,r1GKzP5xx,Recurrent Normalization Propagation,We propose a LSTM parametrization that preser...,"César Laurent, Nicolas Ballas, Pascal Vincent",Invite to Workshop Track,"[4, 6, 6]","[deep learning, optimization]",None,male,3.504240,19.946053
3,3,2017,S1J0E-71l,Surprisal-Driven Feedback in Recurrent Networks,Recurrent neural nets are widely used for pred...,"K, a, m, i, l, , R, o, c, k, i",Reject,"[3, 4, 3]","[unsupervised learning, applications, deep lea...",None,None,4.553473,16.037763
4,4,2017,SJGCiw5gl,Pruning Convolutional Neural Networks for Reso...,We propose a new formulation for pruning convo...,"Pavlo Molchanov, Stephen Tyree, Tero Karras, T...",Accept (Poster),"[6, 7, 9]","[deep learning, transfer learning]",None,male,-25.827705,-37.891772
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24342,7299,2024,1bbPQShCT2,I-PHYRE: Interactive Physical Reasoning,Current evaluation protocols predominantly ass...,,,[],"[intuitive physics, physical reasoning]",None,None,43.137120,44.316133
24343,7300,2024,Ny150AblPu,EXPOSING TEXT-IMAGE INCONSISTENCY USING DIFFUS...,In the battle against widespread online misinf...,,,[],"[mis-contextualization, media forensic]",None,None,59.742172,-22.673627
24344,7301,2024,ZGBOfAQrMl,Video Super-Resolution Transformer with Masked...,"Recently, Vision Transformer has achieved grea...",,,[],"[video super-resolution, adaptive, memory and ...",None,None,57.933273,-3.932825
24345,7302,2024,J2kRjUAOLh,Contrastive Predict-and-Search for Mixed Integ...,Mixed integer linear programs (MILP) are flex...,,,[],[mixed integer programs; contrastive learning],None,None,-11.437999,21.289523


In [ ]:
labels_iclr = np.load(variables_path / "updated_dataset" / "labels_iclr.npy")
colors_iclr = np.load(variables_path / "updated_dataset" / "colors_iclr.npy")

pickle_in = open(
    variables_path / "updated_dataset" / "dict_label_to_color.pkl", "rb"
)
dict_label_to_color = pickle.load(pickle_in)

# kNN accuracy for centered and whitened representations
Using both euclidean and cosine distances for comparison.
The numbers here and in the paper are obtained using an older and smaller version of the ICLR dataset (see shapes in a cell below).

In [ ]:
def print_table(knn_accs):
    print(["euclidean", "cosine"])
    print(
        "Raw:      ",
        round(knn_accs[0, 0] * 100, 1),
        round(knn_accs[0, 1] * 100, 1),
    )
    print(
        "Centered: ",
        round(knn_accs[1, 0] * 100, 1),
        round(knn_accs[1, 1] * 100, 1),
    )
    print(
        "Whitened: ",
        round(knn_accs[2, 0] * 100, 1),
        round(knn_accs[2, 1] * 100, 1),
    )
    print("---------------")

## MPNet

In [ ]:
# load
saving_path = Path("embeddings_" + "mpnet") / Path("updated_dataset")
embedding_av = np.load(
    variables_path / saving_path / "embedding_abstracts_only_av.npy"
)

embedding_asbtracts_only_av_after_training_av_1_epoch = np.load(
    variables_path
    / saving_path
    / "embedding_asbtracts_only_av_after_training_av_1_epoch.npy"
)

In [ ]:
print(embedding_av.shape)
print(embedding_asbtracts_only_av_after_training_av_1_epoch.shape)

(24347, 768)
(24347, 768)


### Baseline

In [ ]:
%%time
knn_accuracy_before_training_centered_and_whitened = (
    knn_accuracy_whitening_scores(
        embedding_av[labels_iclr != "unlabeled"],
        labels_iclr[labels_iclr != "unlabeled"],
        rs=42,
    )
)
knn_accuracy_before_training_centered_and_whitened_rs23 = (
    knn_accuracy_whitening_scores(
        embedding_av[labels_iclr != "unlabeled"],
        labels_iclr[labels_iclr != "unlabeled"],
        rs=23,
    )
)

CPU times: user 50.7 s, sys: 2min 19s, total: 3min 10s
Wall time: 4.99 s


In [ ]:
# rs=23
print_table(knn_accuracy_before_training_centered_and_whitened)
print_table(knn_accuracy_before_training_centered_and_whitened_rs23)

['euclidean', 'cosine']
Raw:       37.4 39.7
Centered:  37.0 35.8
Whitened:  5.5 18.5
---------------
['euclidean', 'cosine']
Raw:       39.8 41.6
Centered:  38.3 37.4
Whitened:  5.5 20.3
---------------


### After fine-tuning

In [ ]:
%%time
knn_accuracy_after_training_centered_and_whitened = (
    knn_accuracy_whitening_scores(
        embedding_asbtracts_only_av_after_training_av_1_epoch[
            labels_iclr != "unlabeled"
        ],
        labels_iclr[labels_iclr != "unlabeled"],
        rs=42,
    )
)
knn_accuracy_after_training_centered_and_whitened_rs23 = (
    knn_accuracy_whitening_scores(
        embedding_asbtracts_only_av_after_training_av_1_epoch[
            labels_iclr != "unlabeled"
        ],
        labels_iclr[labels_iclr != "unlabeled"],
        rs=23,
    )
)

CPU times: user 56.7 s, sys: 2min 36s, total: 3min 33s
Wall time: 5.3 s


In [ ]:
print_table(knn_accuracy_after_training_centered_and_whitened)
print_table(knn_accuracy_after_training_centered_and_whitened_rs23)

['euclidean', 'cosine']
Raw:       58.7 59.3
Centered:  55.7 54.7
Whitened:  36.8 55.0
---------------
['euclidean', 'cosine']
Raw:       60.6 60.2
Centered:  56.5 56.1
Whitened:  36.3 57.9
---------------


In [ ]:
print("BEFORE FINE-TUNING")
print_table(knn_accuracy_before_training_centered_and_whitened)
print("AFTER FINE-TUNING")
print_table(knn_accuracy_after_training_centered_and_whitened)

BEFORE FINE-TUNING
['euclidean', 'cosine']
Raw:       37.4 39.7
Centered:  37.0 35.8
Whitened:  5.5 18.5
---------------
AFTER FINE-TUNING
['euclidean', 'cosine']
Raw:       58.7 59.3
Centered:  55.7 54.7
Whitened:  36.8 55.0
---------------
